In [ ]:
import os, sys

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"
os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import logging
import utils.logger as _log_mod
def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers: return logger
    logger.setLevel(logging.DEBUG)
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S"))
    logger.addHandler(h)
    return logger
_log_mod.setup_logger = setup_logger

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"CWD: {os.getcwd()}")

In [ ]:
!pip install lightning lightgbm --quiet

In [ ]:
"""앙상블 Walk-Forward (체크포인트 로드)
TFT v2 + LightGBM + GARCH 체크포인트를 윈도우별로 로드하여
앙상블 메타모델의 Walk-Forward 성능을 검증.

체크포인트 의존:
  models/tft_v2_wf/checkpoints/window_XX.ckpt
  models/lgbm_wf/checkpoints/window_XX.joblib
  models/garch_wf/checkpoints/window_XX.parquet
"""
import warnings, json, joblib
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression

try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl

warnings.filterwarnings("ignore")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ===== 설정 =====
BASE_PATH = Path(os.environ.get("BASE_PATH"))
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
MODEL_SAVE_PATH = BASE_PATH / "models" / "ensemble_wf"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

# 체크포인트 경로
TFT_CKPT_DIR = BASE_PATH / "models" / "tft_v2_wf" / "checkpoints"
LGBM_CKPT_DIR = BASE_PATH / "models" / "lgbm_wf" / "checkpoints"
GARCH_CKPT_DIR = BASE_PATH / "models" / "garch_wf" / "checkpoints"

WF_START = "2021-01-01"
WF_END = "2026-01-31"
WF_STEP_MONTHS = 3
MAX_ENCODER_LENGTH = 30
BATCH_SIZE = 256

CLEANED_FEATURES = ["vix_change", "vix", "macd_norm", "momentum_20d",
                    "relative_return", "high_low_range", "kospi200_return", "volume_ratio_5d"]

# 윈도우 생성
windows = []
current = pd.Timestamp(WF_START)
end = pd.Timestamp(WF_END)
while current < end:
    test_end = current + pd.DateOffset(months=WF_STEP_MONTHS) - pd.Timedelta(days=1)
    if test_end > end: test_end = end
    train_end = current - pd.Timedelta(days=1)
    windows.append((str(train_end.date()), str(current.date()), str(test_end.date())))
    current += pd.DateOffset(months=WF_STEP_MONTHS)

# 체크포인트 존재 여부 확인
print(f"Walk-Forward 윈도우: {len(windows)}개")
print(f"\n체크포인트 확인:")
for dir_name, ckpt_dir in [("TFT", TFT_CKPT_DIR), ("LightGBM", LGBM_CKPT_DIR), ("GARCH", GARCH_CKPT_DIR)]:
    if ckpt_dir.exists():
        n = len(list(ckpt_dir.iterdir()))
        print(f"  {dir_name}: {n}개 체크포인트 ({ckpt_dir})")
    else:
        print(f"  {dir_name}: 디렉토리 없음! ({ckpt_dir})")

In [ ]:
# ===== 데이터 로드 =====
df = pd.read_parquet(str(TFT_FEATURE_PATH))
df["date"] = pd.to_datetime(df["date"])
df["target_5d"] = df["target_5d"].astype(int)
CLEANED_FEATURES = [c for c in CLEANED_FEATURES if c in df.columns]
N_FEATURES = len(CLEANED_FEATURES)
print(f"데이터: {len(df):,} rows, {N_FEATURES} features")

In [ ]:
# ===== TFT v2 모델 정의 (로드용) =====
class GatedLinearUnit(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc=nn.Linear(d,d); self.gate=nn.Linear(d,d)
    def forward(self, x): return self.fc(x)*torch.sigmoid(self.gate(x))

class GRN(nn.Module):
    def __init__(self, d_in, d_h, d_out, drop=0.1):
        super().__init__()
        self.fc1=nn.Linear(d_in,d_h); self.fc2=nn.Linear(d_h,d_out)
        self.gate=GatedLinearUnit(d_out); self.norm=nn.LayerNorm(d_out)
        self.drop=nn.Dropout(drop); self.skip=nn.Linear(d_in,d_out) if d_in!=d_out else nn.Identity()
    def forward(self, x):
        r=self.skip(x); h=self.drop(F.elu(self.fc2(F.elu(self.fc1(x)))))
        return self.norm(self.gate(h)+r)

class VSN(nn.Module):
    def __init__(self, n_v, d, drop=0.1):
        super().__init__()
        self.grns=nn.ModuleList([GRN(d,d,d,drop) for _ in range(n_v)])
        self.sg=GRN(n_v*d,d,n_v,drop)
    def forward(self, x):
        B,S,V,D=x.shape
        p=torch.stack([self.grns[i](x[:,:,i,:]) for i in range(V)],dim=2)
        w=torch.softmax(self.sg(x.reshape(B,S,V*D)),dim=-1).unsqueeze(-1)
        return (p*w).sum(dim=2)

class TFTv2(pl.LightningModule):
    def __init__(self, n_feat, seq_len=30, d=128, heads=4, n_lstm=1, drop=0.2, n_cls=2, lr=5e-4, cw=None):
        super().__init__()
        self.save_hyperparameters(ignore=["cw"]); self.lr=lr
        self.fe=nn.Linear(1,d); self.vsn=VSN(n_feat,d,drop)
        self.lstm=nn.LSTM(d,d,n_lstm,batch_first=True)
        self.attn=nn.MultiheadAttention(d,heads,dropout=drop,batch_first=True)
        self.an=nn.LayerNorm(d); self.ag=GatedLinearUnit(d)
        self.go=GRN(d,d,d,drop); self.head=nn.Linear(d,n_cls)
        self.loss_fn=nn.CrossEntropyLoss(weight=cw) if cw is not None else nn.CrossEntropyLoss()
    def forward(self, x):
        B,S,F=x.shape; x=self.fe(x.unsqueeze(-1)); x=self.vsn(x)
        x,_=self.lstm(x); a,_=self.attn(x,x,x); x=self.an(x+self.ag(a))
        return self.head(self.go(x[:,-1,:]))
    def training_step(self,b,_): x,y=b; return self.loss_fn(self(x),y)
    def configure_optimizers(self): return torch.optim.AdamW(self.parameters(),lr=self.lr)

print("TFT v2 모델 정의 완료 (로드 전용)")

In [ ]:
# ===== 유틸 함수 =====
def make_tft_samples(full_df, start, end, seq_len, feats):
    """TFT용 시퀀스 샘플 생성 (date, ticker 메타 포함)."""
    samples, metas = [], []
    s=pd.Timestamp(start); e=pd.Timestamp(end) if end else full_df["date"].max()
    for _,g in full_df.groupby("ticker"):
        g=g.sort_values("time_idx"); v=g[feats].values.astype(np.float32)
        t=g["target_5d"].values.astype(np.int64); d=g["date"].values; tk=g["ticker"].values
        for i in range(seq_len,len(g)):
            if d[i]>=s and d[i]<=e:
                samples.append((v[i-seq_len:i],t[i]))
                metas.append({"date": str(d[i])[:10], "ticker": str(tk[i])})
    return samples, metas

class SeqDS(Dataset):
    def __init__(self,s): self.s=s
    def __len__(self): return len(self.s)
    def __getitem__(self,i): x,y=self.s[i]; return torch.tensor(x),torch.tensor(y)

def predict_tft(model, samples):
    """TFT 모델로 예측."""
    loader = DataLoader(SeqDS(samples), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)
    ps, ls = [], []
    model.eval(); model.cuda()
    with torch.no_grad():
        for x,y in loader:
            ps.extend(torch.softmax(model(x.cuda()),dim=-1)[:,1].cpu().numpy()); ls.extend(y.numpy())
    return np.array(ps), np.array(ls)

def predict_lgbm(model, full_df, start, end, feats):
    """LightGBM 모델로 예측."""
    s=pd.Timestamp(start); e=pd.Timestamp(end)
    mask = (full_df["date"]>=s) & (full_df["date"]<=e)
    sub = full_df[mask]
    X = sub[feats].values.astype(np.float32)
    y = sub["target_5d"].values
    probs = model.predict_proba(X)[:, 1]
    metas = [{"date": str(d)[:10], "ticker": t} for d, t in zip(sub["date"].values, sub["ticker"].values)]
    return probs, y, metas

print("유틸 정의 완료")

In [ ]:
# ===== 앙상블 Walk-Forward 실행 =====
wf_results = []
all_methods_results = {"simple_avg": [], "weighted": [], "stacking": [],
                       "tft_only": [], "lgbm_only": []}

for i, (train_end, test_start, test_end) in enumerate(windows):
    print(f"\n{'='*70}")
    print(f"  Window [{i:2d}/{len(windows)}] test={test_start}~{test_end}")
    print(f"{'='*70}")
    
    # 체크포인트 경로
    tft_ckpt = TFT_CKPT_DIR / f"window_{i:02d}_te_{train_end}.ckpt"
    lgbm_ckpt = LGBM_CKPT_DIR / f"window_{i:02d}_te_{train_end}.joblib"
    garch_ckpt = GARCH_CKPT_DIR / f"window_{i:02d}_te_{train_end}.parquet"
    
    # 체크포인트 확인
    missing = []
    if not tft_ckpt.exists(): missing.append("TFT")
    if not lgbm_ckpt.exists(): missing.append("LightGBM")
    if missing:
        print(f"  SKIP: 체크포인트 없음 {missing}")
        wf_results.append({"window": i, "train_end": train_end, "error": f"missing: {missing}"})
        continue
    
    try:
        # 1. TFT 체크포인트 로드 & 예측
        print(f"  TFT 로드: {tft_ckpt.name}")
        tft_model = TFTv2.load_from_checkpoint(str(tft_ckpt), strict=False)
        tft_samples, tft_metas = make_tft_samples(df, test_start, test_end, MAX_ENCODER_LENGTH, CLEANED_FEATURES)
        
        if len(tft_samples) < 100:
            print(f"  SKIP: TFT 샘플 부족 ({len(tft_samples)})")
            wf_results.append({"window": i, "train_end": train_end, "error": "샘플 부족"})
            del tft_model; torch.cuda.empty_cache()
            continue
        
        tft_probs, tft_labels = predict_tft(tft_model, tft_samples)
        del tft_model; torch.cuda.empty_cache()
        
        # 2. LightGBM 체크포인트 로드 & 예측
        print(f"  LightGBM 로드: {lgbm_ckpt.name}")
        lgbm_model = joblib.load(str(lgbm_ckpt))
        lgbm_probs, lgbm_labels, lgbm_metas = predict_lgbm(lgbm_model, df, test_start, test_end, CLEANED_FEATURES)
        del lgbm_model
        
        # 3. GARCH 로드 (변동성 정보)
        garch_df = None
        if garch_ckpt.exists():
            garch_df = pd.read_parquet(str(garch_ckpt))
            print(f"  GARCH 로드: {garch_ckpt.name} ({len(garch_df)} 종목)")
        
        # 4. 예측 매칭 (TFT는 시퀀스, LightGBM은 전체행)
        tft_df = pd.DataFrame(tft_metas)
        tft_df["tft_prob"] = tft_probs
        tft_df["label"] = tft_labels
        
        lgbm_df = pd.DataFrame(lgbm_metas)
        lgbm_df["lgbm_prob"] = lgbm_probs
        
        merged = tft_df.merge(lgbm_df, on=["date", "ticker"], how="inner")
        
        # GARCH 리스크 정보 추가
        if garch_df is not None and "ticker" in garch_df.columns:
            garch_sub = garch_df[["ticker", "vol_5d", "risk_flag"]].copy()
            merged = merged.merge(garch_sub, on="ticker", how="left")
            merged["vol_5d"] = merged["vol_5d"].fillna(merged["vol_5d"].median())
            merged["risk_flag"] = merged["risk_flag"].fillna(0).astype(int)
        
        if len(merged) < 50:
            print(f"  SKIP: 매칭 부족 ({len(merged)})")
            wf_results.append({"window": i, "train_end": train_end, "error": "매칭 부족"})
            continue
        
        y_true = merged["label"].values
        
        # 5. 앙상블 방법들
        # Simple Average
        avg_probs = (merged["tft_prob"] + merged["lgbm_prob"]).values / 2
        # Weighted (TFT 0.6)
        wavg_probs = (merged["tft_prob"] * 0.6 + merged["lgbm_prob"] * 0.4).values
        # Stacking (이전 윈도우로 학습된 메타모델 — 여기선 현재 윈도우 내 CV)
        from sklearn.model_selection import cross_val_predict
        X_meta = merged[["tft_prob", "lgbm_prob"]].values
        meta_lr = LogisticRegression(random_state=42, class_weight="balanced")
        try:
            stack_probs = cross_val_predict(meta_lr, X_meta, y_true, cv=3, method="predict_proba")[:, 1]
        except:
            stack_probs = avg_probs  # fallback
        
        # 6. 평가
        results = {"window": i, "train_end": train_end, "test_start": test_start,
                   "test_end": test_end, "n_samples": len(merged)}
        
        for name, probs in [("tft_only", merged["tft_prob"].values),
                            ("lgbm_only", merged["lgbm_prob"].values),
                            ("simple_avg", avg_probs),
                            ("weighted", wavg_probs),
                            ("stacking", stack_probs)]:
            preds = (probs >= 0.5).astype(int)
            da = accuracy_score(y_true, preds)
            try: auc = roc_auc_score(y_true, probs)
            except: auc = float("nan")
            results[f"{name}_da"] = round(da * 100, 2)
            results[f"{name}_auc"] = round(auc, 4)
            all_methods_results[name].append(da * 100)
        
        wf_results.append(results)
        
        # 요약 출력
        print(f"  TFT={results['tft_only_da']:.1f}% | LGBM={results['lgbm_only_da']:.1f}% | "
              f"Avg={results['simple_avg_da']:.1f}% | W.Avg={results['weighted_da']:.1f}% | "
              f"Stack={results['stacking_da']:.1f}%")
        
    except Exception as e:
        print(f"  ERROR: {e}")
        wf_results.append({"window": i, "train_end": train_end, "error": str(e)})
        torch.cuda.empty_cache()

print(f"\n앙상블 Walk-Forward 완료: {len(wf_results)}개 윈도우")

In [ ]:
# ===== 결과 요약 =====
valid = [r for r in wf_results if "error" not in r]

if valid:
    print("="*80)
    print("  앙상블 Walk-Forward 결과 비교")
    print("="*80)
    print(f"  윈도우: {len(valid)}개 / {len(wf_results)}개\n")
    
    # 방법별 평균 DA
    print(f"  {'방법':20s} {'평균 DA':>10s} {'Std':>8s} {'Min':>8s} {'Max':>8s}")
    print(f"  {'-'*56}")
    for name, label in [("tft_only", "TFT v2 단독"), ("lgbm_only", "LightGBM 단독"),
                        ("simple_avg", "Simple Average"), ("weighted", "Weighted (0.6/0.4)"),
                        ("stacking", "Stacking (LR CV)")]:
        das = all_methods_results[name]
        if das:
            print(f"  {label:20s} {np.mean(das):>9.2f}% {np.std(das):>7.2f} {min(das):>7.2f}% {max(das):>7.2f}%")
    
    # 최적 방법
    best_method = max(all_methods_results, key=lambda k: np.mean(all_methods_results[k]) if all_methods_results[k] else 0)
    best_label = {"tft_only": "TFT v2", "lgbm_only": "LightGBM", "simple_avg": "Simple Avg",
                  "weighted": "Weighted", "stacking": "Stacking"}[best_method]
    print(f"\n  최적 방법: {best_label} (평균 DA={np.mean(all_methods_results[best_method]):.2f}%)")
    
    # 윈도우별 상세
    print(f"\n  {'#':>3} {'기간':>24} {'TFT':>7} {'LGBM':>7} {'Avg':>7} {'W.Avg':>7} {'Stack':>7} {'최고':>8}")
    print(f"  {'-'*76}")
    for r in valid:
        methods_da = {"TFT": r["tft_only_da"], "LGBM": r["lgbm_only_da"],
                      "Avg": r["simple_avg_da"], "W.Avg": r["weighted_da"], "Stack": r["stacking_da"]}
        best = max(methods_da, key=methods_da.get)
        print(f"  [{r['window']:2d}] {r['test_start']}~{r['test_end']} "
              f"{r['tft_only_da']:6.1f} {r['lgbm_only_da']:6.1f} "
              f"{r['simple_avg_da']:6.1f} {r['weighted_da']:6.1f} {r['stacking_da']:6.1f}  {best}")
    print("="*80)
else:
    print("유효한 결과 없음")

In [ ]:
# ===== 저장 =====
json.dump(wf_results, open(str(MODEL_SAVE_PATH / f"ensemble_wf_results_{datetime.now().strftime('%Y%m%d')}.json"), "w"),
          ensure_ascii=False, indent=2, default=str)

# 방법별 요약도 저장
summary = {}
for name in all_methods_results:
    das = all_methods_results[name]
    if das:
        summary[name] = {"mean_da": round(np.mean(das), 2), "std": round(np.std(das), 2),
                         "min": round(min(das), 2), "max": round(max(das), 2)}
json.dump(summary, open(str(MODEL_SAVE_PATH / "ensemble_summary.json"), "w"), indent=2)
print(f"저장: {MODEL_SAVE_PATH}")

## 앙상블 Walk-Forward (체크포인트 로드)\n\n### 구조\n```\n각 윈도우:\n  TFT 체크포인트 로드 (수초) → 예측\n  LightGBM 체크포인트 로드 (수초) → 예측\n  GARCH 체크포인트 로드 (수초) → 변동성\n  → 3가지 앙상블 방법 비교\n```\n\n### 앙상블 방법\n1. **Simple Average**: (TFT + LightGBM) / 2\n2. **Weighted Average**: TFT*0.6 + LightGBM*0.4\n3. **Stacking**: LogisticRegression (3-fold CV)\n\n### 비교 기준\n- TFT WF 평균 DA: 51.07%\n- LightGBM WF 평균 DA: 52.30%\n- 앙상블이 개별 모델보다 높으면 성공